# Income Pipeline, Part 1: Flat File (IRS Tax Data by ZIP Code)

**Goal:** prepare IRS income data by ZIP code so it can be merged with web and API sources in part 4.

**Data:** IRS Statistics of Income for 2010 tax returns (`Data/2010 Income Tax.csv`): 167,022 rows and 73 coded columns, with each ZIP code split into six income brackets.

**Steps:** keep 7 useful columns and give them clear names, remove 306 state-total and combined rows that are not real ZIP codes, add the six income brackets together, restore leading zeros, and calculate average wages per return.

**Result:** one row for each of 27,786 ZIP codes, covering 137.1 million returns and $5.68 trillion in salaries and wages, saved as `Data/Income_flat.csv`.

**Tools:** Python (pandas)

**Context:** master's coursework (DSC540 term project), cleaned up and corrected for this portfolio. This is part of a four-notebook pipeline: 1 flat file, 2 web scraping, 3 API, 4 merge and visualize.

## 1. Load the IRS file

In [1]:
import pandas as pd

# IRS Statistics of Income, 2010 individual tax returns by ZIP code
df = pd.read_csv('Data/2010 Income Tax.csv')
print(df.shape)
df[['STATE', 'zipcode', 'agi_stub', 'N1', 'MARS2', 'NUMDEP', 'A00200']].head()

(167022, 73)


,STATE,zipcode,agi_stub,N1,MARS2,NUMDEP,A00200
0,AL,0,1,912574.0,137786.0,506532.0,9033252.0
1,AL,0,2,484831.0,167221.0,336103.0,14163028.0
2,AL,0,3,249919.0,154478.0,185041.0,12072207.0
3,AL,0,4,152015.0,123748.0,129887.0,10270849.0
4,AL,0,5,164628.0,147038.0,158345.0,16811858.0


## 2. Keep the columns needed and give them clear names
The IRS uses codes for its columns. Counts start with N and dollar amounts start with A; **dollar amounts are in thousands**.

In [2]:
columns = {'STATE': 'State', 'zipcode': 'ZipCode', 'agi_stub': 'Income_Bracket',
           'N1': 'Num_Returns', 'MARS2': 'Num_Joint_Returns', 'NUMDEP': 'Num_Dependents',
           'A00200': 'Salaries_Wages_Thousands', 'A04800': 'Taxable_Income_Thousands',
           'A59660': 'Earned_Income_Credit_Thousands'}
df = df[list(columns)].rename(columns=columns)
print('Missing values:', df.isna().sum().sum(), '  Duplicate rows:', df.duplicated().sum())

Missing values: 0   Duplicate rows: 0


## 3. Remove summary rows and combine the income brackets
- ZIP code 00000 holds each state's totals, and 99999 combines small ZIP codes the IRS does not list separately; neither is a real ZIP code, so both are removed.
- Each ZIP code is split into six adjusted gross income brackets, so the brackets are added together to give one row per ZIP code.

In [3]:
print('Rows per ZIP code (income brackets):', df.groupby('ZipCode').size().mode()[0])
summary_rows = df['ZipCode'].isin([0, 99999])
print(f"State-total and combined rows removed: {summary_rows.sum():,}")
df = df[~summary_rows]

income = df.drop(columns='Income_Bracket').groupby(['ZipCode', 'State'], as_index=False).sum()
income['ZipCode'] = income['ZipCode'].astype(str).str.zfill(5)    # restore leading zeros, e.g. 2108 -> 02108
income['Avg_Wages_Per_Return'] = (income['Salaries_Wages_Thousands'] * 1000 / income['Num_Returns']).round(0)
print(f"ZIP codes: {len(income):,}   Duplicate ZIP codes: {income['ZipCode'].duplicated().sum()}")
income.head()

Rows per ZIP code (income brackets): 6
State-total and combined rows removed: 306
ZIP codes: 27,786   Duplicate ZIP codes: 0


,ZipCode,State,Num_Returns,Num_Joint_Returns,Num_Dependents,Salaries_Wages_Thousands,Taxable_Income_Thousands,Earned_Income_Credit_Thousands,Avg_Wages_Per_Return
0,01001,MA,8608.0000,3136.0000,3933.0000,334941.0000,290323.0000,1531.0000,38910.0
1,01002,MA,10166.0000,3458.0000,4642.0000,412511.0000,466738.0000,1764.0000,40578.0
2,01005,MA,2177.0000,959.0000,1350.0000,92412.0000,77309.0000,426.0000,42449.0
3,01007,MA,7144.0000,3255.0000,4392.0000,352293.0000,297677.0000,1199.0000,49313.0
4,01008,MA,624.0001,313.0001,301.0001,27123.0001,24075.0001,82.0001,43466.0


## 4. Quick checks

In [4]:
print(f"Total returns: {income['Num_Returns'].sum():,.0f}")
print(f"Total salaries and wages: ${income['Salaries_Wages_Thousands'].sum() / 1e6:,.1f} billion")
print('States (incl. DC):', income['State'].nunique())
print('ZIP codes per state, top 5:'); print(income['State'].value_counts().head().to_string())
print('Highest average wages per return:')
print(income.nlargest(5, 'Avg_Wages_Per_Return')[['ZipCode', 'State', 'Num_Returns', 'Avg_Wages_Per_Return']].to_string(index=False))

Total returns: 137,148,399
Total salaries and wages: $5,678.3 billion
States (incl. DC): 51
ZIP codes per state, top 5:
State
TX    1623
NY    1547
CA    1486
PA    1369
IL    1229
Highest average wages per return:
ZipCode State  Num_Returns  Avg_Wages_Per_Return
  94027    CA    3169.0000              351137.0
  07931    NJ    1542.0000              314474.0
  90067    CA    3123.0000              305346.0
  10007    NY    3591.0000              302129.0
  89413    NV     179.0002              282117.0


In [5]:
income.to_csv('Data/Income_flat.csv', index=False)
print(f'Saved Data/Income_flat.csv: {len(income):,} rows (one per ZIP code)')

Saved Data/Income_flat.csv: 27,786 rows (one per ZIP code)
